In [ ]:
# -*- coding: utf-8 -*-
import sys
import os
import gc
import json
import h5py
import pandas as pd
import numpy as np
from tqdm import tqdm

if __name__ == "__main__":
    # --- PATH SUMBER DATA ASLI ---
    CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
    HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
    ZHI_GENG_JSON = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Benchmark_ STEAD 3C_ test n15275 r100/STEAD data, test n15275 r100.json'
    
    # --- PATH OUTPUT BERKAS MATANG BARU ---
    PREPROCESSED_H5 = '/Volumes/Extreme SSD/stream_stead/data_stead/stead_processed_100k_core.h5'
    
    # Pastikan direktori output sudah terbentuk
    os.makedirs(os.path.dirname(PREPROCESSED_H5), exist_ok=True)

    # --------------------------------------------------------------------------
    # FASE 1: SAMPLING & ANTI-LEAKAGE FILTERING (DATA BALANCING)
    # --------------------------------------------------------------------------
    print("[INFO] Mengekstraksi daftar hitam anti-leakage dari JSON...")
    with open(ZHI_GENG_JSON, 'r') as f:
        zhi_geng_data = json.load(f)
    zhi_geng_traces = set(zhi_geng_data.keys()) if isinstance(zhi_geng_data, dict) else set(zhi_geng_data)
        
    print("[INFO] Memuat metadata CSV STEAD...")
    df_raw = pd.read_csv(CSV_PATH, low_memory=False)
    df_filtered = df_raw[df_raw['trace_category'].isin(['earthquake_local', 'noise'])]
    df_unseen = df_filtered[~df_filtered['trace_name'].isin(zhi_geng_traces)]
    
    df_eq = df_unseen[df_unseen['trace_category'] == 'earthquake_local']
    df_noise = df_unseen[df_unseen['trace_category'] == 'noise']
    
    # Mengunci kuota seimbang (50.000 Gempa & 50.000 Noise)
    n_samples = 50000 
    df_eq_sample = df_eq.sample(n=min(n_samples, len(df_eq)), random_state=42)
    df_noise_sample = df_noise.sample(n=min(n_samples, len(df_noise)), random_state=42)
    
    # Penggabungan dan pengacakan urutan baris secara homogen
    df_final = pd.concat([df_eq_sample, df_noise_sample]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    # SHIELD MEMORY ANTI-LEAK: Konversi DataFrame menjadi array NumPy datar
    trace_names = df_final['trace_name'].to_numpy()
    trace_categories = df_final['trace_category'].to_numpy()
    p_arrivals = df_final['p_arrival_sample'].fillna(0).to_numpy().astype(np.int32)
    
    del df_raw, df_filtered, df_unseen, df_eq, df_noise, df_eq_sample, df_noise_sample, df_final
    gc.collect()

    # --------------------------------------------------------------------------
    # FASE 2: EKSEKUSI PIPELINE PRAPEMROSESAN & FREEZING DATA (KOREKSI AXIS=0)
    # --------------------------------------------------------------------------
    num_points = 700          # Jendela potongan model: 7 detik (100 Hz)
    norm_points = 900         # Jendela normalisasi: 9 detik pasca P-arrival
    total_target = len(trace_names)

    print(f"\n[INFO] Membuat file berkas biner matang baru di -> {PREPROCESSED_H5}")
    
    # Membuka berkas HDF5 baru dengan fitur kompresi gzip tingkat tinggi
    with h5py.File(PREPROCESSED_H5, 'w') as f_out:
        # Inisialisasi struktur matriks biner kosong siap isi untuk 1C, 3C, dan Label
        dset_1c = f_out.create_dataset("X_1C", shape=(total_target, num_points, 1), dtype=np.float32, compression="gzip", compression_opts=4)
        dset_3c = f_out.create_dataset("X_3C", shape=(total_target, num_points, 3), dtype=np.float32, compression="gzip", compression_opts=4)
        dset_y  = f_out.create_dataset("Y", shape=(total_target,), dtype=np.int32)
        
        # Simpan metadata nama trace untuk keperluan audit atau pelacakan stasiun
        dset_names = f_out.create_dataset("trace_name", shape=(total_target,), dtype=h5py.string_dtype(encoding='utf-8'))

        with h5py.File(HDF5_PATH, 'r') as f_in:
            data_group = f_in['data']
            
            for idx in tqdm(range(total_target), desc="Freezing STEAD 100K (1C & 3C)"):
                try:
                    trace_id = trace_names[idx]
                    category = trace_categories[idx]
                    
                    if trace_id not in data_group:
                        continue
                        
                    # 1. Tarik komponen penuh 3C (E, N, Z) langsung dari SSD tanpa filter frekuensi
                    raw_wave_3c = data_group[trace_id][()]
                    
                    # 2. Detrending secara terisolasi per sumbu pada sinyal kontinu mentah
                    detrended_wave_3c = raw_wave_3c - np.mean(raw_wave_3c, axis=0)
                    
                    if category == 'earthquake_local':
                        p_arrival = p_arrivals[idx]
                        start_idx = int(p_arrival)
                        end_idx = start_idx + num_points
                        norm_end_idx = start_idx + norm_points
                        
                        if norm_end_idx > len(detrended_wave_3c) or start_idx < 0: 
                            continue
                            
                        # Slicing tepat di P-arrival
                        wave_slice_3c = detrended_wave_3c[start_idx:end_idx, :]
                        
                        # [KOREKSI BAPAK] Hitung pembagi skala pada jendela 9 detik independen per saluran (axis=0)
                        norm_val_3c = np.max(np.abs(detrended_wave_3c[start_idx:norm_end_idx, :]), axis=0)
                        true_label = 1
                    else:
                        # Kelas Noise: Data lingkungan sebelum P-arrival
                        wave_slice_3c = detrended_wave_3c[:num_points, :]
                        
                        # Hitung pembagi skala pada 9 detik awal independen per saluran (axis=0)
                        norm_val_3c = np.max(np.abs(detrended_wave_3c[:norm_points, :]), axis=0)
                        true_label = 0
                    
                    # Proteksi pembagian nol menggunakan vektorisasi array
                    norm_val_3c[norm_val_3c == 0] = 1e-8
                        
                    # 3. Normalisasi amplitudo maksimum secara independen berkat fitur broadcasting Numpy
                    wave_slice_3c /= norm_val_3c
                    
                    # 4. Isolasi komponen vertikal murni (sumbu Z / indeks ke-2) untuk dataset 1C
                    # Karena wave_slice_3c sudah dinormalisasi per saluran, kita otomatis mendapatkan
                    # komponen Z yang skalanya akurat dan independen dari anomali saluran E atau N.
                    wave_slice_1c = wave_slice_3c[:, 2].reshape(num_points, 1)
                    
                    # 5. Tanamkan matriks numerik yang sudah matang ke dalam dataset HDF5 baru
                    dset_1c[idx] = wave_slice_1c
                    dset_3c[idx] = wave_slice_3c
                    dset_y[idx]  = true_label
                    dset_names[idx] = trace_id
                    
                except Exception:
                    continue

    print("\n=======================================================")
    print(" [SUKSES] BERKAS BINER PRE-DUMP STEAD 100K BERHASIL DICETAK!")
    print("=======================================================")
    print(f"File Target: {PREPROCESSED_H5}")
    print("Isi Dataset:")
    print(" - 'X_1C'  : Tensor matriks vertikal murni (100000, 700, 1)")
    print(" - 'X_3C'  : Tensor matriks tiga komponen  (100000, 700, 3)")
    print(" - 'Y'     : Array label ground truth      (100000,)")
    print("=======================================================")

[INFO] Mengekstraksi daftar hitam anti-leakage dari JSON...
[INFO] Memuat metadata CSV STEAD...

[INFO] Membuat file berkas biner matang baru di -> /Volumes/Extreme SSD/stream_stead/data_stead/stead_processed_100k_core.h5


Freezing STEAD 100K (1C & 3C):   8%|▊         | 7700/100000 [26:34<9:11:58,  2.79it/s] Exception ignored in: <function WeakValueDictionary.__init__.<locals>.remove at 0x106291ea0>
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniforge/base/envs/mcu_quake_env/lib/python3.10/weakref.py", line 106, in remove
    def remove(wr, selfref=ref(self), _atomic_removal=_remove_dead_weakref):
KeyboardInterrupt: 
Freezing STEAD 100K (1C & 3C):   8%|▊         | 7782/100000 [27:05<9:50:35,  2.60it/s] 